# Library

In [ ]:
import pandas as pd
import re
import ast
import os
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import SnowballStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns

: 

# Load Datasets that have been collected from three genres

In this section, the game datasets that have been collected are loaded, after which lowercase letters are applied in the genre section to filter your genre by removing sub-genres that are not related to adventure, casual and sport.

In [ ]:
# Load adventure game datasets
game_datasets = pd.read_csv('../data/adventure_games_data.csv')

game_datasets['genres'] = game_datasets['genres'].str.lower()
adventure_games = game_datasets[game_datasets['genres'].str.contains('adventure')]
adventure_games['genres'] = 'adventure'

# Save the datasets with new file csv
adventure_games.to_csv('../data/adventure_games_genre.csv', index=False)

# Show several datasets
adventure_games


In [ ]:
# Load casual game datasets
game_datasets = pd.read_csv('../data/casual_games_data.csv')

game_datasets['genres'] = game_datasets['genres'].str.lower()
casual_games = game_datasets[game_datasets['genres'].str.contains('casual')]
casual_games['genres'] = 'casual'

# Save the datasets with new file csv
casual_games.to_csv('../data/casual_games_genre.csv', index=False)

# Show several datasets
casual_games


In [ ]:
# Load sport game datasets
game_datasets = pd.read_csv('../data/sports_games_data.csv')

game_datasets['genres'] = game_datasets['genres'].str.lower()
sport_games = game_datasets[game_datasets['genres'].str.contains('sport')]
sport_games['genres'] = 'sport'

# Save the datasets with new file csv
sport_games.to_csv('../data/sports_games_genre.csv', index=False)

# Show several file
sport_games


# Combine file

After filtering the genre by leaving only the main genre, in this section, we combine the three genres and delete several games that are considered duplicates based on their titles. After that, the final results of the datasets that will be used are seen.

In [ ]:
# Load Datasets
adventure_games = pd.read_csv('../data/adventure_games_genre.csv')
casual_games = pd.read_csv('../data/casual_games_genre.csv')
sport_games = pd.read_csv('../data/sports_games_genre.csv')

combined_games = pd.concat([adventure_games, casual_games, sport_games], ignore_index=True)
combined_games = combined_games.drop_duplicates(subset='title')

# Save combined file
combined_games.to_csv('../data/combined_games_genre.csv', index=False)

# Showing final datasets for Preprocessing
combined_games


In [ ]:
# Load Datasets
game_datasets = pd.read_csv('../data/combined_games_genre.csv')

# View overall data totals
total_games = len(game_datasets)
print(f"Total games that have been collected: {total_games}")

# Counting the number of games per genre
genre_counts = game_datasets['genres'].value_counts()

print("\nNumber of games per genre:")
print(genre_counts)

# Preprocessing

In this session, preprocessing of datasets is carried out so that it can be more accurate when classifying game genres by performing case folding, Stopword Removal, Tokenizing, and Stemming using Stemmer Snowball.

> Case Folding

the process of changing all letters in a text to lowercase to ensure consistency.

In [ ]:
def clean_text(game_datasets, text_field, new_text_field_name):
    # Convert to lowercase and handle non-string values
    game_datasets[new_text_field_name] = game_datasets[text_field].apply(lambda x: str(x).lower() if isinstance(x, str) else '')
    game_datasets[new_text_field_name] = game_datasets[new_text_field_name].apply(
        lambda elem: re.sub(r"(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)|^rt|http.?", "", elem)
    )
    game_datasets[new_text_field_name] = game_datasets[new_text_field_name].apply(lambda elem: re.sub(r"\d+", "", elem))
    return game_datasets

In [ ]:
# Clean the text for 'title', 'genres', and 'description' columns again with the updated function
game_datasets_clean = clean_text(game_datasets, 'title', 'title_clean')
game_datasets_clean = clean_text(game_datasets_clean, 'genres', 'genre_clean')
game_datasets_clean = clean_text(game_datasets_clean, 'description', 'description_clean')

In [ ]:
# Display the cleaned dataframe
game_datasets_clean.head(10)

> Stopword Removal

the process of removing common words that have no significant meaning in text analysis, such as "and," "the," or "is."

In [ ]:
nltk.download('stopwords')
stop = stopwords.words('english')

In [ ]:
# Define a function to remove stopwords
def remove_stopwords_from_text(text):
    if isinstance(text, str):
        return ' '.join([word for word in text.split() if word.lower() not in stop])
    return text

# Apply the function to 'title_clean', 'genre_clean', and 'description_clean' columns in game_datasets_clean
game_datasets_clean['title_clean_StopWord'] = game_datasets_clean['title_clean'].apply(remove_stopwords_from_text)
game_datasets_clean['genre_clean_StopWord'] = game_datasets_clean['genre_clean'].apply(remove_stopwords_from_text)
game_datasets_clean['description_clean_StopWord'] = game_datasets_clean['description_clean'].apply(remove_stopwords_from_text)


In [ ]:
# Display the first 10 rows
game_datasets_clean.head(10)

> Tokenizing

the process of breaking down text into smaller units, such as words or sentences, called tokens.

In [ ]:
nltk.download('punkt')

In [ ]:
# Filling empty values ​​in 'title_clean_StopWord' and 'description_clean_StopWord' columns with empty strings to avoid errors
game_datasets_clean['title_clean_StopWord'] = game_datasets_clean['title_clean_StopWord'].fillna('')
game_datasets_clean['description_clean_StopWord'] = game_datasets_clean['description_clean_StopWord'].fillna('')

# Tokenization of 'title_clean_StopWord' and 'description_clean_StopWord' columns
game_datasets_clean['title_tokens'] = game_datasets_clean['title_clean_StopWord'].apply(lambda x: re.findall(r'\b\w+\b', x))
game_datasets_clean['description_tokens'] = game_datasets_clean['description_clean_StopWord'].apply(lambda x: re.findall(r'\b\w+\b', x))


In [ ]:
# Display the first 10 rows
game_datasets_clean.head(10)

> Stemming using Snowball Stemmer

the process of changing a word into its basic form (stem) using the Snowball Stemmer algorithm. Snowball Stemmer is designed to cut off the endings of words in certain languages, such as English, so that words that have the same meaning but in different forms can be treated consistently.

In [ ]:
from nltk.stem import SnowballStemmer
snowball = SnowballStemmer("english")

In [ ]:
# Applying stemming to 'title_tokens' and 'description_tokens' columns
game_datasets_clean['title_snowball'] = game_datasets_clean['title_tokens'].apply(lambda tokens: [snowball.stem(token) for token in tokens])
game_datasets_clean['description_snowball'] = game_datasets_clean['description_tokens'].apply(lambda tokens: [snowball.stem(token) for token in tokens])

In [ ]:
game_datasets_clean.head(10)

In [ ]:
game_datasets_clean.to_csv('../data/preprocessing_result.csv', index= False)

In [ ]:
game_datasets.head(10)

# Weighting using TF-IDF

a technique for assigning weight to each word in a document based on the frequency of its occurrence, either within a particular document (term frequency) or across a collection of documents (inverse document frequency).

- Term Frequency (TF) measures how often a word appears in a single document. Words that appear frequently in a document get a higher TF score.

- Inverse Document Frequency (IDF) measures how unique or specific a word is across documents. Words that appear frequently in many documents get a lower IDF score, because they are considered less important.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Load Datasets
file_path = '../data/preprocessing_result.csv'
df = pd.read_csv(file_path)

In [ ]:
# Reading dataset
print(f"Dataset loaded: {len(df)} rows")
print(df.columns)

In [ ]:
# Safe converter
def safe_join(x):
    try:
        return ' '.join(ast.literal_eval(x)) if isinstance(x, str) else ''
    except:
        return ''

In [ ]:
# Convert columns
df['description_snowball'] = df['description_snowball'].apply(safe_join)
df['title_snowball'] = df['title_snowball'].apply(safe_join)

In [ ]:
# Combine text
df['combined_text'] = df['title_snowball'] + " " + df['description_snowball']
df['combined_text'] = df['combined_text'].fillna('')

In [ ]:
# Checking if a text column exists in a dataset
print("\nSample text:")
print(df['combined_text'].head())

In [ ]:
# TfidfVectorizer Initialization
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.85,      
    min_df=2,         
    ngram_range=(1, 2)
)

In [ ]:
# Transform text → vector
tfidf_matrix = tfidf_vectorizer.fit_transform(df['combined_text'])

print(f"\nTF-IDF Shape: {tfidf_matrix.shape}")

In [ ]:
feature_names = tfidf_vectorizer.get_feature_names_out()

In [ ]:
# Function to display the term with the highest TF-IDF score in a document.
def display_top_terms(tfidf_row, features, top_n=10):
    nz_indices = tfidf_row.nonzero()[1]
    nz_scores = tfidf_row.data

    if len(nz_scores) == 0:
        return []

    sorted_idx = nz_scores.argsort()[::-1]
    top_indices = nz_indices[sorted_idx[:top_n]]
    top_scores = nz_scores[sorted_idx[:top_n]]

    return [(features[i], score) for i, score in zip(top_indices, top_scores)]

In [ ]:
# Displays top terms for the first document
doc_id = min(10, tfidf_matrix.shape[0] - 1)

tfidf_row = tfidf_matrix[doc_id]
top_terms = display_top_terms(tfidf_row, feature_names)

print(f"\nTop terms for document {doc_id}:")

for term, score in top_terms:
    print(f"{term}: {score:.4f}")

In [ ]:
# Calculate the total TF-IDF score for each term in the corpus.
term_sums = tfidf_matrix.sum(axis=0).A1

terms_freq_df = pd.DataFrame({
    'term': feature_names,
    'tfidf_sum': term_sums
}).sort_values(by='tfidf_sum', ascending=False)

print("\nTop 20 global terms:")
print(terms_freq_df.head(20))

# Implementing Naive Bayes

the process of applying the Naive Bayes algorithm to data classification based on the principle of probability. Naive Bayes is a simple but effective model that assumes that each feature (word or token in the text) is independent of each other in determining the class or label, an assumption called "naive independence."

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [ ]:
# Text column names and labels
text_column = 'description_snowball'
label_column = 'genre_clean_StopWord'

# Checking if a text column exists in a dataset
if text_column not in df.columns:
    print(f"Column '{text_column}' not found in dataset.")
    print("Please ensure the dataset has processed text columns..")
else:
    print(f"\nExample data from columns '{text_column}':")
    print(df[text_column].head())
    print(f"\nExample data from columns '{label_column}':")
    print(df[label_column].value_counts())


In [ ]:
# Features (X) and labels (y)
X = df[text_column]
y = df[label_column]

In [ ]:
# TfidfVectorizer Initialization
tfidf_vectorizer = TfidfVectorizer()

# Convert text to numeric features
X_tfidf = tfidf_vectorizer.fit_transform(X)

print(f"\nThe text has been converted into a TF-IDF vector with shapes {X_tfidf.shape}")

In [ ]:
# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nAmount of training data: {X_train.shape[0]}")
print(f"Amount of testing data: {X_test.shape[0]}")

In [ ]:
# Initialization of Multinomial Naive Bayes model
model = MultinomialNB()

# Train Model
model.fit(X_train, y_train)

In [ ]:
# Predict labels on test data
y_pred = model.predict(X_test)

In [ ]:
# Calculating accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel accuracy: {accuracy * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

conf_matrix = confusion_matrix(y_test, y_pred)
labels = ['Adventure', 'Casual', 'Sports']

# Print the confusion matrix to console
print("\nConfusion Matrix:")
print(conf_matrix)

# Plotting the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Showing classification report
class_report = classification_report(y_test, y_pred)
print("\nClassification Report:")
print(class_report)


# Improve using Pipeline and GridSearchCV

Using **Pipeline** and **GridSearchCV** together is expected to improve model efficiency and accuracy by automating the process of data transformation and optimal hyperparameter search. Pipeline organizes the workflow into one integrated step—from text vectorization using TF-IDF to classification with Naive Bayes—ensuring consistency between training and testing data. Meanwhile, GridSearchCV tests various combinations of hyperparameters, such as n-gram size and smoothing parameters in Naive Bayes, to find the best settings based on cross-validation results. This combination not only reduces the potential for manual errors in data processing but also improves model performance by automatically optimizing parameters.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Merge the title and description columns that have gone through the stemming process
game_datasets['combined_text'] = game_datasets['title_snowball'].apply(lambda tokens: ' '.join(tokens)) + " " + game_datasets['description_snowball'].apply(lambda tokens: ' '.join(tokens))


In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    game_datasets['combined_text'],
    game_datasets['genre_clean_StopWord'],
    test_size=0.2,
    random_state=42
)

In [ ]:
# Performing TF-IDF transformation on X_train and X_test
tfidf_vectorizer = TfidfVectorizer()
X_train = tfidf_vectorizer.fit_transform(X_train_text)
X_test = tfidf_vectorizer.transform(X_test_text)

In [ ]:
print(X_test_text.head())

In [ ]:
# Define a pipeline with TF-IDF and Naive Bayes
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

In [ ]:
# Define the parameter grid for GridSearch
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],    # Unigram and Bigram
    'tfidf__max_df': [0.7, 0.85],              # Max document frequency to ignore very common words
    'tfidf__min_df': [2, 5],                   # Min document frequency to ignore rare words
    'nb__alpha': [0.5, 1.0]                    # Smoothing parameter for Naive Bayes
}

In [ ]:
# Use GridSearchCV to find the best parameters
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_text, y_train)

In [ ]:
# Showing the best parameters and best accuracy of Cross-Validation
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("Best Parameters and Cross-Validation Results:\n")
print(f"- Best Parameters:")
print(f"  - nb__alpha: {best_params['nb__alpha']}")
print(f"  - tfidf__max_df: {best_params['tfidf__max_df']}")
print(f"  - tfidf__min_df: {best_params['tfidf__min_df']}")
print(f"  - tfidf__ngram_range: {best_params['tfidf__ngram_range']}")
print(f"\n- Best Cross-Validation Accuracy: {best_score:.4f}")

In [ ]:
# Predict on the test set
y_pred = grid_search.predict(X_test_text)
test_accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", test_accuracy)

In [ ]:
# Making predictions on test data (test set)
y_pred = grid_search.predict(X_test_text)

# Creating a confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred, labels=grid_search.classes_)

# Displaying confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=grid_search.classes_, yticklabels=grid_search.classes_)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix for Test Data')
plt.show()

# Displays classification report for test data
print("Classification Report for Test Data:")
print(classification_report(y_test, y_pred))


In [ ]:
#Create a DataFrame to compare predicted results with actual data.
results_df = pd.DataFrame({
    'Title and Description': X_test_text.values,
    'Actual Genre': y_test.values,
    'Predicted Genre': y_pred
})

print("Comparison of Title, Description, Actual Genre, and Predicted Genre:")
print(results_df.head(10))

In [ ]:
# Genre Prediction Function for New Data
def predict_genre(title, description):
    combined_text = title + " " + description
    predicted_genre = grid_search.predict([combined_text])
    return predicted_genre[0]

# Example of Genre Prediction on New Data
title_input = "Cosmic Hoops"
description_input = "highly detailed real-time sumulation allows players to feel every muscle twitch and heartbeat as thes race against the clock and opponents in a dynamic, ever changing environment"
predicted_genre = predict_genre(title_input, description_input)
print("Predicted Genre:", predicted_genre)

In [ ]:
# Genre Prediction Function for New Data
def predict_genre(title, description):
    combined_text = title + " " + description
    predicted_genre = grid_search.predict([combined_text])
    return predicted_genre[0]

# Example of Genre Prediction on New Data
title_input = "Cat Cafe"
description_input = "menage the most unique cat cafe in town! take care of cute cats, make all kinds of delicious food and drinks, and decorate the cafe with adorable furniture to attract costumers"
predicted_genre = predict_genre(title_input, description_input)
print("Predicted Genre:", predicted_genre)

In [ ]:
best_model = grid_search.best_estimator_

# Save model
joblib.dump(best_model, "../models/model.pkl")